# Bagging

**Notebook 2 of 5.** Twenty trees that disagree by about 5 MPa, each worse than the single
tree they were meant to improve on. This notebook averages them, and the average beats all
twenty.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.metrics import root_mean_squared_error

## 1. The data

`data/concrete.csv` again: 1,030 recipes, seven ingredients, an age in days, and
`compressive_strength` in MPa. Same split, same `random_state=42`, so the scores compare
with notebook 01's.

In [ ]:
df = pd.read_csv("data/concrete.csv")

X = df.drop(columns="compressive_strength")
y = df["compressive_strength"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"{len(X_train)} training mixes, {len(X_test)} test mixes")

## 2. The twenty trees again

The same function as notebook 01: one bootstrap sample per tree, one row of test predictions
per tree.

In [ ]:
def bootstrap_predictions(max_depth=None, n_trees=20, seed=42):
    """Fit one tree per bootstrap sample of the training set and predict the test set.

    Args:
        max_depth: depth limit passed to every tree; None means no limit.
        n_trees: how many bootstrap samples, and so how many trees.
        seed: seed for the random generator that draws the samples.

    Returns:
        array of shape (n_trees, number of test mixes): one row of predictions per tree.
    """
    rng = np.random.default_rng(seed)
    n = len(X_train)
    predictions = []
    for _ in range(n_trees):
        sample = rng.integers(0, n, size=n)

        # iloc, because a bootstrap sample is a list of positions, repeats included
        tree = DecisionTreeRegressor(max_depth=max_depth, random_state=42)
        tree.fit(X_train.iloc[sample], y_train.iloc[sample])
        predictions.append(tree.predict(X_test))
    return np.array(predictions)


deep = bootstrap_predictions()

# the tree of notebook 01, trained on all 824 mixes rather than a resample
single = DecisionTreeRegressor(random_state=42).fit(X_train, y_train)

per_tree = [root_mean_squared_error(y_test, row) for row in deep]
print(f"single tree, all the data: {root_mean_squared_error(y_test, single.predict(X_test)):.2f} MPa")
print(f"the twenty, on average:    {np.mean(per_tree):.2f} MPa")

Each bootstrap tree saw about 63% of the distinct mixes, so each is worse than the tree that
saw all of them: **8.11 against 7.13**. Twenty worse models, and the next cell makes one
better one out of them.

## 3. Average them

> **Exercise:**
>
> Average the twenty predictions for each test mix and compute the RMSE of that average.
> Compare it with the 8.11 above and with the single tree's 7.13.

<details><summary>
Click here for a hint...
</summary>

`deep` has one row per tree, so `deep.mean(axis=0)` averages down the trees and leaves one
prediction per test mix.

</details>

In [ ]:
# your code: average the twenty predictions per mix, then score the average

<details><summary>
Click here to compare your answer...
</summary>

**5.91 MPa**, against the single tree's 7.13 and the best of the twenty at 7.06. The average
is not a compromise between the trees; it beats all of them.

</details>

## 4. Why averaging works

Each prediction is the truth plus an error, and the errors come from which rows that tree
drew. They point in different directions, so averaging cancels much of them and leaves the
part the trees agree on.

> **Bagging = Bootstrap + Aggregating.** One model per bootstrap sample, then the **mean**
> for regression or a **majority vote** for classification.

Two conditions, and the second is where bagging fails:

- the models are each **better than guessing**;
- their errors are **different from each other**. Twenty identical trees average to one tree.

The bootstrap exists to supply the second.

## 5. How many trees?

Depth has a best value, past which a tree gets worse. The number of trees is not like that.

> **Exercise:**
>
> Fit 200 bootstrap trees, then score the average of the first 1, 2, 5, 10, 25, 50, 100 and
> 200 of them. Plot RMSE against the number of trees. Where does it stop improving?

<details><summary>
Click here for a hint...
</summary>

`bootstrap_predictions(n_trees=200)` once, then `big[:k].mean(axis=0)` for each `k`: no need
to refit for every count.

</details>

In [ ]:
# your code: 200 trees, then the RMSE of the first 1, 2, 5, 10, 25, 50, 100 and 200

<details><summary>
Click here to compare your answer...
</summary>

**7.74 with one tree, 5.54 with two hundred**, most of it in by fifty: 6.02 at ten, 5.65 at
fifty, 5.57 at a hundred.

The curve flattens and does not turn back up. **More trees never makes bagging worse**, it
only stops helping, so the cost of another hundred is time rather than accuracy. It is the
one setting here you can pick by patience.

</details>

## 6. The library version

`BaggingRegressor` does what sections 2 and 3 did by hand: draw the samples, fit an estimator
to each, average their predictions.

In [ ]:
bagging = BaggingRegressor(
    DecisionTreeRegressor(random_state=42),
    n_estimators=200,
    random_state=42,
    n_jobs=-1,          # fit the trees in parallel; they do not depend on each other
)
bagging.fit(X_train, y_train)

print(f"BaggingRegressor: {root_mean_squared_error(y_test, bagging.predict(X_test)):.2f} MPa")

**5.52 against the 5.54 built by hand**, the difference being only which rows each draw
picked.

`BaggingClassifier` votes instead of averaging. Neither is limited to trees, but trees are
the usual choice: low bias and high variance is exactly what bagging needs.

## 7. What bagging cannot fix

Averaging removes the error that changes from sample to sample, and nothing of the error
every tree shares.

In [ ]:
shallow = bootstrap_predictions(max_depth=3)

print(f"depth-3 trees, on average:  {np.mean([root_mean_squared_error(y_test, r) for r in shallow]):.2f} MPa")
print(f"their average:              {root_mean_squared_error(y_test, shallow.mean(axis=0)):.2f} MPa")
print(f"deep trees, their average:  {root_mean_squared_error(y_test, deep.mean(axis=0)):.2f} MPa")

Bagging the depth-3 trees improves them from 11.14 to **10.19**, and stops. They are all too
simple in the same direction, and averaging cannot remove a shared error.

The rule that follows:

- **bag models with low bias and high variance** — deep, unpruned trees;
- **do not prune first.** Pruning lowers the variance averaging would have removed, and
  raises the bias it cannot.

## Summary

- **Bagging = bootstrap + aggregating**: one model per sample, then the mean or a vote.
- Each bootstrap tree is **worse** than a tree on all the data, 8.11 against 7.13; their
  average is **better than any of them**, 5.91.
- It works because the errors differ and partly cancel.
- **More trees never hurts**: the curve flattens near fifty and stays flat.
- It removes variance, not bias: the depth-3 trees stay at 10.19.

Every one of these trees still searched all eight features at every split. Next, in
`03_random_forests.ipynb`: what happens when you take that away too.